In [ ]:
# ================================================================
# PNT1 LOGISTICS DIGITAL TWIN
# QUANTUM MACHINE LEARNING (QML) ROUTE IMPACT CLASSIFICATION
#
# ONE-CELL GOOGLE COLAB NOTEBOOK
#
# INPUT:
#   PNT1 logistics Excel dataset
#
# OUTPUT:
#   QML model
#   Classical baselines
#   Confusion matrix
#   ROC-AUC
#   PR-AUC
#   Accuracy / Precision / Recall / F1
#   Specificity / Sensitivity / NPV
#   MCC / Cohen's Kappa
#   Log Loss / Brier Score
#   95% Bootstrap Confidence Intervals
#   Training convergence
#   ROC curve
#   Precision-Recall curve
#   Model comparison
#   Runtime comparison
#   Feature importance
#   Saved models + CSV results + PNG graphs + ZIP
#
# IMPORTANT:
# The supplied workbook is synthetic data.
# Therefore scenario labels are generated from explicit operational
# rules and are NOT real-world measured ground truth.
# ================================================================


# ================================================================
# 1. INSTALL PACKAGES
# ================================================================

import sys
import subprocess
import os
import glob
import time
import json
import zipfile
import warnings

warnings.filterwarnings("ignore")

packages = [
    "pennylane",
    "openpyxl",
    "scikit-learn",
    "joblib"
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q"
] + packages)


# ================================================================
# 2. IMPORTS
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    matthews_corrcoef,
    cohen_kappa_score,
    log_loss,
    brier_score_loss
)

import joblib

import pennylane as qml
from pennylane import numpy as qnp


# ================================================================
# 3. RANDOM SEEDS
# ================================================================

SEED = 42

np.random.seed(SEED)
rng = np.random.default_rng(SEED)


# ================================================================
# 4. LOAD EXCEL DATASET
# ================================================================

print("=" * 70)
print("UPLOAD / LOAD PNT1 LOGISTICS DATASET")
print("=" * 70)

try:

    from google.colab import files

    excel_files = glob.glob("/content/*.xlsx")

    if len(excel_files) == 0:

        print("Please upload your PNT1 Excel dataset.")

        uploaded = files.upload()

        excel_files = glob.glob("/content/*.xlsx")

    DATA_PATH = excel_files[0]

except Exception:

    excel_files = (
        glob.glob("/content/*.xlsx")
        +
        glob.glob("/mnt/data/*.xlsx")
    )

    if len(excel_files) == 0:

        raise FileNotFoundError(
            "No Excel dataset found. Please upload the PNT1 dataset."
        )

    DATA_PATH = excel_files[0]


print("Dataset:", DATA_PATH)


# ================================================================
# 5. READ SHEETS
# ================================================================

xls = pd.ExcelFile(DATA_PATH)

print("\nAvailable sheets:")
for s in xls.sheet_names:
    print(" -", s)

required_sheets = [
    "Transport Network",
    "Disruptions",
    "Orders Shipments"
]

missing = [
    s for s in required_sheets
    if s not in xls.sheet_names
]

if missing:

    raise ValueError(
        "Missing required sheets: "
        + ", ".join(missing)
    )


transport = pd.read_excel(
    DATA_PATH,
    sheet_name="Transport Network"
)

disruptions = pd.read_excel(
    DATA_PATH,
    sheet_name="Disruptions"
)

orders = pd.read_excel(
    DATA_PATH,
    sheet_name="Orders Shipments"
)


# ================================================================
# 6. CLEAN COLUMN NAMES
# ================================================================

for df in [
    transport,
    disruptions,
    orders
]:

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )


# ================================================================
# 7. VALIDATE REQUIRED COLUMNS
# ================================================================

transport_required = [
    "Route_ID",
    "Origin_Node_ID",
    "Destination_Node_ID",
    "Mode",
    "Distance_km",
    "Travel_Time_hr",
    "Cost_per_Unit",
    "Capacity_Units_per_Day",
    "Availability",
    "Risk_Score"
]

disruption_required = [
    "Disruption_ID",
    "Disruption_Type",
    "Affected_Node_or_Route_ID",
    "Duration_Hours",
    "Severity_1_to_5",
    "Capacity_Reduction_pct",
    "Scenario_Status"
]

order_required = [
    "Order_ID",
    "Shipment_ID",
    "Origin_Node_ID",
    "Destination_Node_ID",
    "SKU",
    "Quantity_Units",
    "Route_ID",
    "Shipment_Status",
    "Priority"
]

for col in transport_required:

    if col not in transport.columns:

        raise ValueError(
            f"Transport Network missing column: {col}"
        )

for col in disruption_required:

    if col not in disruptions.columns:

        raise ValueError(
            f"Disruptions missing column: {col}"
        )

for col in order_required:

    if col not in orders.columns:

        raise ValueError(
            f"Orders Shipments missing column: {col}"
        )


# ================================================================
# 8. NUMERIC CLEANING
# ================================================================

numeric_transport = [
    "Distance_km",
    "Travel_Time_hr",
    "Cost_per_Unit",
    "Capacity_Units_per_Day",
    "Risk_Score"
]

for col in numeric_transport:

    transport[col] = pd.to_numeric(
        transport[col],
        errors="coerce"
    )

numeric_disruption = [
    "Duration_Hours",
    "Severity_1_to_5",
    "Capacity_Reduction_pct"
]

for col in numeric_disruption:

    disruptions[col] = pd.to_numeric(
        disruptions[col],
        errors="coerce"
    )


# Remove unusable transport rows
transport = transport.dropna(
    subset=[
        "Route_ID",
        "Origin_Node_ID",
        "Destination_Node_ID",
        "Distance_km",
        "Travel_Time_hr",
        "Cost_per_Unit",
        "Capacity_Units_per_Day",
        "Risk_Score"
    ]
).copy()


disruptions = disruptions.dropna(
    subset=[
        "Disruption_ID",
        "Disruption_Type",
        "Affected_Node_or_Route_ID",
        "Duration_Hours",
        "Severity_1_to_5",
        "Capacity_Reduction_pct"
    ]
).copy()


# Clip sensible values
transport["Risk_Score"] = (
    transport["Risk_Score"]
    .clip(0, 1)
)

disruptions["Severity_1_to_5"] = (
    disruptions["Severity_1_to_5"]
    .clip(1, 5)
)

disruptions["Capacity_Reduction_pct"] = (
    disruptions["Capacity_Reduction_pct"]
    .clip(0, 100)
)


# ================================================================
# 9. DATASET SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("PNT1 DATASET SUMMARY")
print("=" * 70)

print("Transport routes :", transport["Route_ID"].nunique())
print("Disruptions      :", len(disruptions))
print("Orders           :", len(orders))
print("Transport rows   :", len(transport))

print("\nTransport modes:")
print(transport["Mode"].value_counts())

print("\nDisruption types:")
print(disruptions["Disruption_Type"].value_counts())

print("\nShipment statuses:")
print(orders["Shipment_Status"].value_counts())


# ================================================================
# 10. BUILD SCENARIO DATASET
#
# We combine:
# route characteristics
# +
# disruption characteristics
#
# Target:
#   0 = VIABLE
#   1 = IMPACTED
#
# A route is impacted when the disruption directly targets:
#   - the route
#   - the route origin
#   - the route destination
# or creates a severe network condition.
# ================================================================

scenario_rows = []

route_records = transport.to_dict("records")
disruption_records = disruptions.to_dict("records")


# ------------------------------------------------
# Normal scenarios
# ------------------------------------------------

for route in route_records:

    for repeat in range(8):

        distance = float(
            route["Distance_km"]
        )

        travel = float(
            route["Travel_Time_hr"]
        )

        cost = float(
            route["Cost_per_Unit"]
        )

        capacity = float(
            route["Capacity_Units_per_Day"]
        )

        risk = float(
            route["Risk_Score"]
        )

        # Small natural operational variation
        noise_distance = (
            distance
            * rng.normal(
                1.0,
                0.01
            )
        )

        noise_travel = (
            travel
            * rng.normal(
                1.0,
                0.025
            )
        )

        noise_cost = (
            cost
            * rng.normal(
                1.0,
                0.02
            )
        )

        noise_capacity = (
            capacity
            * rng.normal(
                1.0,
                0.02
            )
        )

        congestion = np.clip(
            rng.normal(
                0.20,
                0.06
            ),
            0,
            1
        )

        reliability = np.clip(
            1.0
            - risk
            - rng.normal(
                0.03,
                0.015
            ),
            0,
            1
        )

        scenario_rows.append({

            "Route_ID":
                str(route["Route_ID"]),

            "Mode":
                str(route["Mode"]),

            "Target_Type":
                "NONE",

            "Disruption_Type":
                "NONE",

            "Distance_km":
                max(
                    0.01,
                    noise_distance
                ),

            "Travel_Time_hr":
                max(
                    0.01,
                    noise_travel
                ),

            "Cost_per_Unit":
                max(
                    0.001,
                    noise_cost
                ),

            "Capacity_Units_per_Day":
                max(
                    1,
                    noise_capacity
                ),

            "Base_Risk":
                risk,

            "Operational_Risk":
                np.clip(
                    risk
                    + rng.normal(
                        0,
                        0.02
                    ),
                    0,
                    1
                ),

            "Reliability":
                reliability,

            "Congestion":
                congestion,

            "Severity":
                0.0,

            "Capacity_Reduction":
                0.0,

            "Duration_Hours":
                0.0,

            "Route_Target_Flag":
                0,

            "Origin_Target_Flag":
                0,

            "Destination_Target_Flag":
                0,

            "Label":
                0
        })


# ------------------------------------------------
# Disruption scenarios
# ------------------------------------------------

for route in route_records:

    route_id = str(
        route["Route_ID"]
    )

    origin = str(
        route["Origin_Node_ID"]
    )

    destination = str(
        route["Destination_Node_ID"]
    )

    for disruption in disruption_records:

        disruption_target = str(
            disruption[
                "Affected_Node_or_Route_ID"
            ]
        )

        disruption_type = str(
            disruption[
                "Disruption_Type"
            ]
        )

        severity_raw = float(
            disruption[
                "Severity_1_to_5"
            ]
        )

        reduction_raw = float(
            disruption[
                "Capacity_Reduction_pct"
            ]
        )

        duration = float(
            disruption[
                "Duration_Hours"
            ]
        )

        route_match = int(
            disruption_target == route_id
        )

        origin_match = int(
            disruption_target == origin
        )

        destination_match = int(
            disruption_target == destination
        )

        # Route directly targeted:
        # highest impact.
        #
        # Origin/destination affected:
        # medium/high impact depending on severity.
        #
        # Other route/network:
        # lower propagation impact.

        if route_match:

            target_type = "ROUTE"

            target_strength = 1.00

        elif origin_match:

            target_type = "ORIGIN_NODE"

            target_strength = 0.85

        elif destination_match:

            target_type = "DESTINATION_NODE"

            target_strength = 0.85

        else:

            target_type = "OTHER"

            target_strength = 0.08


        severity_norm = (
            severity_raw / 5.0
        )

        capacity_reduction = (
            reduction_raw / 100.0
        )

        # Operational impact score.
        #
        # This is the scenario generator,
        # not a learned model.

        impact_score = (

            0.38
            * severity_norm
            * target_strength

            +

            0.32
            * capacity_reduction
            * target_strength

            +

            0.12
            * min(
                duration / 120.0,
                1.0
            )
            * target_strength

            +

            0.10
            * float(
                route["Risk_Score"]
            )

            +

            0.08
            * rng.random()
        )

        impacted = int(
            impact_score >= 0.34
        )

        # Operational variables after disruption
        travel_multiplier = (

            1.0

            +

            0.30
            * severity_norm
            * target_strength

            +

            0.45
            * capacity_reduction
            * target_strength
        )

        travel_time = (

            float(
                route["Travel_Time_hr"]
            )

            * travel_multiplier

            * rng.normal(
                1.0,
                0.02
            )
        )

        congestion = np.clip(

            0.18

            +

            0.50
            * severity_norm
            * target_strength

            +

            0.35
            * capacity_reduction
            * target_strength

            +

            0.08
            * rng.random(),

            0,
            1
        )

        operational_risk = np.clip(

            float(
                route["Risk_Score"]
            )

            +

            0.22
            * severity_norm
            * target_strength

            +

            0.18
            * capacity_reduction
            * target_strength

            +

            rng.normal(
                0,
                0.015
            ),

            0,
            1
        )

        reliability = np.clip(

            1.0

            -

            operational_risk

            -

            0.20
            * severity_norm
            * target_strength

            -

            0.15
            * capacity_reduction
            * target_strength

            +

            rng.normal(
                0,
                0.015
            ),

            0,
            1
        )

        effective_capacity_ratio = max(

            0.0,

            1.0
            -
            capacity_reduction
            * target_strength
        )

        scenario_rows.append({

            "Route_ID":
                route_id,

            "Mode":
                str(route["Mode"]),

            "Target_Type":
                target_type,

            "Disruption_Type":
                disruption_type,

            "Distance_km":
                max(
                    0.01,
                    float(
                        route["Distance_km"]
                    )
                    * rng.normal(
                        1,
                        0.01
                    )
                ),

            "Travel_Time_hr":
                max(
                    0.01,
                    travel_time
                ),

            "Cost_per_Unit":
                max(
                    0.001,
                    float(
                        route["Cost_per_Unit"]
                    )
                    * (
                        1
                        +
                        0.15
                        * capacity_reduction
                        * target_strength
                    )
                    * rng.normal(
                        1,
                        0.02
                    )
                ),

            "Capacity_Units_per_Day":
                max(
                    1,
                    float(
                        route[
                            "Capacity_Units_per_Day"
                        ]
                    )
                    * effective_capacity_ratio
                ),

            "Base_Risk":
                float(
                    route["Risk_Score"]
                ),

            "Operational_Risk":
                operational_risk,

            "Reliability":
                reliability,

            "Congestion":
                congestion,

            "Severity":
                severity_norm,

            "Capacity_Reduction":
                capacity_reduction
                * target_strength,

            "Duration_Hours":
                duration
                * target_strength,

            "Route_Target_Flag":
                route_match,

            "Origin_Target_Flag":
                origin_match,

            "Destination_Target_Flag":
                destination_match,

            "Label":
                impacted
        })


scenario_df = pd.DataFrame(
    scenario_rows
)


# ================================================================
# 11. DERIVED FEATURES
# ================================================================

scenario_df["Delay_Ratio"] = np.clip(

    scenario_df["Travel_Time_hr"]
    /
    np.maximum(
        transport["Travel_Time_hr"]
        .mean(),
        0.01
    ),

    0,
    50
)

scenario_df["Capacity_Pressure"] = (
    1
    -
    (
        scenario_df[
            "Capacity_Units_per_Day"
        ]
        /
        np.maximum(
            transport[
                "Capacity_Units_per_Day"
            ].max(),
            1
        )
    )
)

scenario_df["Risk_x_Severity"] = (
    scenario_df["Operational_Risk"]
    *
    scenario_df["Severity"]
)

scenario_df["Congestion_x_Severity"] = (
    scenario_df["Congestion"]
    *
    scenario_df["Severity"]
)


print("\n" + "=" * 70)
print("GENERATED DIGITAL-TWIN SCENARIO DATA")
print("=" * 70)

print(
    "Total scenario samples:",
    len(scenario_df)
)

print("\nTarget distribution:")

target_counts = (
    scenario_df["Label"]
    .value_counts()
    .sort_index()
)

for label, count in target_counts.items():

    name = (
        "VIABLE"
        if label == 0
        else "IMPACTED"
    )

    print(
        f"{name:12s}: {count}"
    )


# ================================================================
# 12. FEATURES
# ================================================================

numeric_features = [

    "Distance_km",

    "Travel_Time_hr",

    "Cost_per_Unit",

    "Capacity_Units_per_Day",

    "Base_Risk",

    "Operational_Risk",

    "Reliability",

    "Congestion",

    "Severity",

    "Capacity_Reduction",

    "Duration_Hours",

    "Route_Target_Flag",

    "Origin_Target_Flag",

    "Destination_Target_Flag",

    "Delay_Ratio",

    "Capacity_Pressure",

    "Risk_x_Severity",

    "Congestion_x_Severity"
]

categorical_features = [

    "Mode",

    "Target_Type",

    "Disruption_Type"
]

all_features = (
    numeric_features
    +
    categorical_features
)


X = scenario_df[
    all_features
].copy()

y = scenario_df[
    "Label"
].astype(int)

groups = scenario_df[
    "Route_ID"
].astype(str)


# ================================================================
# 13. HANDLE CATEGORICAL FEATURES
# ================================================================

try:

    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )

except TypeError:

    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )


preprocessor = ColumnTransformer(

    transformers=[

        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),

        (
            "categorical",
            encoder,
            categorical_features
        )
    ]
)


X_processed = preprocessor.fit_transform(
    X
)

print(
    "\nProcessed feature shape:",
    X_processed.shape
)


# ================================================================
# 14. GROUP-AWARE TRAIN / TEST SPLIT
#
# Entire route IDs are kept separate.
# This prevents the same route from appearing
# in both training and testing.
# ================================================================

unique_groups = groups.unique()

if len(unique_groups) >= 4:

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.25,
        random_state=SEED
    )

    train_idx, test_idx = next(

        splitter.split(
            X_processed,
            y,
            groups=groups
        )
    )

else:

    raise ValueError(
        "Not enough unique routes for a group-aware test split."
    )


X_train = X_processed[
    train_idx
]

X_test = X_processed[
    test_idx
]

y_train = y.iloc[
    train_idx
].to_numpy()

y_test = y.iloc[
    test_idx
].to_numpy()

groups_train = groups.iloc[
    train_idx
]

groups_test = groups.iloc[
    test_idx
]


print("\n" + "=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

print(
    "Training samples:",
    len(X_train)
)

print(
    "Testing samples :",
    len(X_test)
)

print(
    "Unseen test routes:",
    sorted(
        groups_test.unique()
    )
)


if len(np.unique(y_train)) < 2:

    raise ValueError(
        "Training set contains only one class. "
        "Please rerun with a different random_state."
    )

if len(np.unique(y_test)) < 2:

    raise ValueError(
        "Test set contains only one class. "
        "Please rerun with a different random_state."
    )


# ================================================================
# 15. PCA FOR QUANTUM ENCODING
#
# Reduce the feature vector to 4 dimensions
# because we use 4 qubits.
# ================================================================

N_QUBITS = 4

pca = PCA(
    n_components=N_QUBITS,
    random_state=SEED
)

X_train_pca = pca.fit_transform(
    X_train
)

X_test_pca = pca.transform(
    X_test
)

print("\n" + "=" * 70)
print("PCA")
print("=" * 70)

print(
    "Explained variance:",
    np.round(
        pca.explained_variance_ratio_,
        4
    )
)

print(
    "Cumulative variance:",
    round(
        np.sum(
            pca.explained_variance_ratio_
        ),
        4
    )
)


# ================================================================
# 16. QUANTUM ANGLE ENCODING
# ================================================================

max_abs = np.maximum(

    np.max(
        np.abs(
            X_train_pca
        ),
        axis=0
    ),

    1e-8
)

angle_scale = (
    np.pi
    /
    (
        1.25
        *
        max_abs
    )
)

X_train_q = np.clip(

    X_train_pca
    *
    angle_scale,

    -np.pi,
    np.pi
)

X_test_q = np.clip(

    X_test_pca
    *
    angle_scale,

    -np.pi,
    np.pi
)


# ================================================================
# 17. QML CIRCUIT
#
# Variational Quantum Classifier
# 4 qubits
# 2 trainable layers
# ================================================================

N_LAYERS = 2

device = qml.device(
    "default.qubit",
    wires=N_QUBITS
)


@qml.qnode(
    device,
    interface="autograd"
)
def qml_circuit(
    x,
    weights
):

    qml.AngleEmbedding(
        x,
        wires=range(
            N_QUBITS
        ),
        rotation="Y"
    )

    for layer in range(
        N_LAYERS
    ):

        for qubit in range(
            N_QUBITS
        ):

            qml.Rot(
                weights[
                    layer,
                    qubit,
                    0
                ],

                weights[
                    layer,
                    qubit,
                    1
                ],

                weights[
                    layer,
                    qubit,
                    2
                ],

                wires=qubit
            )

        for qubit in range(
            N_QUBITS - 1
        ):

            qml.CNOT(
                wires=[
                    qubit,
                    qubit + 1
                ]
            )

        qml.CNOT(
            wires=[
                N_QUBITS - 1,
                0
            ]
        )

    return qml.expval(
        qml.PauliZ(0)
    )


# ================================================================
# 18. QML TRAINING
# ================================================================

rng_qml = np.random.default_rng(
    123
)

weights = qnp.array(

    0.05
    *
    rng_qml.normal(
        size=(
            N_LAYERS,
            N_QUBITS,
            3
        )
    ),

    requires_grad=True
)

scale_param = qnp.array(
    1.0,
    requires_grad=True
)

bias_param = qnp.array(
    0.0,
    requires_grad=True
)


# Calculate class weights
n0 = max(
    1,
    np.sum(
        y_train == 0
    )
)

n1 = max(
    1,
    np.sum(
        y_train == 1
    )
)

weight0 = len(
    y_train
) / (
    2 * n0
)

weight1 = len(
    y_train
) / (
    2 * n1
)


X_train_q_np = [
    qnp.array(
        x,
        requires_grad=False
    )
    for x in X_train_q
]

y_train_q = qnp.array(
    y_train,
    requires_grad=False
)


def qml_loss(
    weights_,
    scale_,
    bias_,
    X_data,
    y_data
):

    circuit_outputs = qnp.stack([

        qml_circuit(
            x,
            weights_
        )

        for x in X_data

    ])

    logits = (
        scale_
        *
        circuit_outputs
        +
        bias_
    )

    probabilities = (
        1
        /
        (
            1
            +
            qnp.exp(
                -logits
            )
        )
    )

    eps = 1e-7

    per_sample_loss = (

        -

        (
            y_data
            *
            qnp.log(
                probabilities
                + eps
            )
            *
            weight1
        )

        -

        (
            (1 - y_data)
            *
            qnp.log(
                1 - probabilities
                + eps
            )
            *
            weight0
        )
    )

    return qnp.mean(
        per_sample_loss
    )


optimizer = qml.AdamOptimizer(
    stepsize=0.05
)

EPOCHS = 25

loss_history = []

print("\n" + "=" * 70)
print("QML TRAINING")
print("=" * 70)

qml_train_start = time.perf_counter()


for epoch in range(
    EPOCHS
):

    (
        weights,
        scale_param,
        bias_param
    ) = optimizer.step(

        lambda w, s, b:

        qml_loss(
            w,
            s,
            b,
            X_train_q_np,
            y_train_q
        ),

        weights,
        scale_param,
        bias_param
    )

    current_loss = float(

        qml_loss(
            weights,
            scale_param,
            bias_param,
            X_train_q_np,
            y_train_q
        )
    )

    loss_history.append(
        current_loss
    )

    if (
        epoch == 0
        or
        (epoch + 1) % 5 == 0
    ):

        print(
            "Epoch",
            f"{epoch+1:02d}/{EPOCHS}",
            "| Loss:",
            f"{current_loss:.6f}"
        )


qml_train_time = (
    time.perf_counter()
    -
    qml_train_start
)


# ================================================================
# 19. QML PREDICTION FUNCTION
# ================================================================

def qml_predict_probability(
    X_data
):

    outputs = []

    start = time.perf_counter()

    for x in X_data:

        quantum_output = float(

            qml_circuit(
                qnp.array(
                    x,
                    requires_grad=False
                ),
                weights
            )

        )

        logit = (

            float(
                scale_param
            )
            *
            quantum_output

            +

            float(
                bias_param
            )
        )

        probability = (

            1
            /
            (
                1
                +
                np.exp(
                    -logit
                )
            )
        )

        outputs.append(
            probability
        )

    elapsed = (
        time.perf_counter()
        - start
    )

    per_sample_ms = (

        elapsed
        /
        max(
            len(X_data),
            1
        )
        *
        1000
    )

    return (
        np.asarray(
            outputs
        ),
        per_sample_ms
    )


qml_prob, qml_inference_ms = (
    qml_predict_probability(
        X_test_q
    )
)

qml_pred = (
    qml_prob >= 0.5
).astype(int)


# ================================================================
# 20. CLASSICAL BASELINE MODELS
# ================================================================

models = {

    "Logistic Regression":
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=SEED
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        ),

    "SVM (RBF)":
        SVC(
            probability=True,
            class_weight="balanced",
            random_state=SEED
        )
}


predictions = {
    "QML-VQC":
        qml_pred
}

probabilities = {
    "QML-VQC":
        qml_prob
}

train_times = {
    "QML-VQC":
        qml_train_time
}

inference_times = {
    "QML-VQC":
        qml_inference_ms
}


# ================================================================
# 21. TRAIN CLASSICAL MODELS
# ================================================================

for name, model in models.items():

    print(
        "\nTraining:",
        name
    )

    start = time.perf_counter()

    model.fit(
        X_train_pca,
        y_train
    )

    train_time = (
        time.perf_counter()
        - start
    )

    start = time.perf_counter()

    probability = (
        model.predict_proba(
            X_test_pca
        )[:, 1]
    )

    inference_time = (

        (
            time.perf_counter()
            - start
        )
        /
        max(
            len(X_test_pca),
            1
        )
        *
        1000
    )

    prediction = (
        probability >= 0.5
    ).astype(int)

    predictions[name] = (
        prediction
    )

    probabilities[name] = (
        probability
    )

    train_times[name] = (
        train_time
    )

    inference_times[name] = (
        inference_time
    )


# ================================================================
# 22. METRICS
# ================================================================

def safe_roc_auc(
    y_true,
    y_prob
):

    try:

        return roc_auc_score(
            y_true,
            y_prob
        )

    except:

        return np.nan


def safe_pr_auc(
    y_true,
    y_prob
):

    try:

        return average_precision_score(
            y_true,
            y_prob
        )

    except:

        return np.nan


def calculate_full_metrics(
    name,
    y_true,
    y_pred,
    y_prob
):

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = (
        cm.ravel()
    )

    sensitivity = (
        tp
        /
        max(
            tp + fn,
            1
        )
    )

    specificity = (
        tn
        /
        max(
            tn + fp,
            1
        )
    )

    npv = (
        tn
        /
        max(
            tn + fn,
            1
        )
    )

    false_positive_rate = (
        fp
        /
        max(
            fp + tn,
            1
        )
    )

    false_negative_rate = (
        fn
        /
        max(
            fn + tp,
            1
        )
    )

    return {

        "Model":
            name,

        "Accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "Precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "Recall_Sensitivity":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "Specificity":
            specificity,

        "NPV":
            npv,

        "F1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "ROC_AUC":
            safe_roc_auc(
                y_true,
                y_prob
            ),

        "PR_AUC":
            safe_pr_auc(
                y_true,
                y_prob
            ),

        "MCC":
            matthews_corrcoef(
                y_true,
                y_pred
            ),

        "Cohen_Kappa":
            cohen_kappa_score(
                y_true,
                y_pred
            ),

        "FPR":
            false_positive_rate,

        "FNR":
            false_negative_rate,

        "Log_Loss":
            log_loss(
                y_true,
                np.vstack(
                    [
                        1 - y_prob,
                        y_prob
                    ]
                ).T,
                labels=[0, 1]
            ),

        "Brier_Score":
            brier_score_loss(
                y_true,
                y_prob
            ),

        "Train_Time_s":
            train_times[name],

        "Inference_ms_per_sample":
            inference_times[name]
    }


metrics_rows = []

for name in probabilities:

    metrics_rows.append(

        calculate_full_metrics(
            name,
            y_test,
            predictions[name],
            probabilities[name]
        )
    )


metrics_df = pd.DataFrame(
    metrics_rows
)


# ================================================================
# 23. PRINT RESULTS
# ================================================================

print("\n" + "=" * 100)
print("COMPLETE MODEL PERFORMANCE")
print("=" * 100)

display(
    metrics_df.round(4)
)


# ================================================================
# 24. QML CLASSIFICATION REPORT
# ================================================================

print("\n" + "=" * 70)
print("QML CLASSIFICATION REPORT")
print("=" * 70)

print(

    classification_report(

        y_test,

        qml_pred,

        labels=[0, 1],

        target_names=[
            "VIABLE",
            "IMPACTED"
        ],

        digits=4,

        zero_division=0
    )
)


# ================================================================
# 25. QML CONFUSION MATRIX
# ================================================================

qml_cm = confusion_matrix(
    y_test,
    qml_pred,
    labels=[0, 1]
)

TN, FP, FN, TP = (
    qml_cm.ravel()
)

qml_sensitivity = (
    TP / max(TP + FN, 1)
)

qml_specificity = (
    TN / max(TN + FP, 1)
)

qml_ppv = (
    TP / max(TP + FP, 1)
)

qml_npv = (
    TN / max(TN + FN, 1)
)

print("\n" + "=" * 70)
print("QML CONFUSION MATRIX COUNTS")
print("=" * 70)

print("True Negative :", TN)
print("False Positive:", FP)
print("False Negative:", FN)
print("True Positive :", TP)

print("\nSensitivity :", round(
    qml_sensitivity,
    4
))

print("Specificity :", round(
    qml_specificity,
    4
))

print("PPV         :", round(
    qml_ppv,
    4
))

print("NPV         :", round(
    qml_npv,
    4
))


# ================================================================
# 26. NORMALIZED CONFUSION MATRIX
# ================================================================

row_totals = np.maximum(
    qml_cm.sum(
        axis=1,
        keepdims=True
    ),
    1
)

qml_cm_normalized = (
    qml_cm
    /
    row_totals
)


# ================================================================
# 27. BOOTSTRAP 95% CONFIDENCE INTERVALS
# ================================================================

def bootstrap_metric_ci(
    metric_fn,
    y_true,
    y_pred,
    iterations=1500
):

    rng_boot = np.random.default_rng(
        1234
    )

    scores = []

    n = len(
        y_true
    )

    for _ in range(
        iterations
    ):

        indices = rng_boot.integers(
            0,
            n,
            n
        )

        yt = y_true[
            indices
        ]

        yp = y_pred[
            indices
        ]

        try:

            score = metric_fn(
                yt,
                yp
            )

            if np.isfinite(score):

                scores.append(
                    float(score)
                )

        except:

            pass

    if len(scores) < 20:

        return (
            np.nan,
            np.nan,
            np.nan
        )

    low, high = np.percentile(
        scores,
        [2.5, 97.5]
    )

    return (
        float(
            np.mean(scores)
        ),
        float(low),
        float(high)
    )


bootstrap_metrics = {

    "Accuracy":
        accuracy_score,

    "Balanced Accuracy":
        balanced_accuracy_score,

    "Precision":
        lambda a, b:
            precision_score(
                a,
                b,
                zero_division=0
            ),

    "Sensitivity":
        lambda a, b:
            recall_score(
                a,
                b,
                zero_division=0
            ),

    "Specificity":
        lambda a, b: (

            confusion_matrix(
                a,
                b,
                labels=[0, 1]
            )[0, 0]

            /

            max(
                confusion_matrix(
                    a,
                    b,
                    labels=[0, 1]
                )[0].sum(),
                1
            )
        ),

    "F1":
        lambda a, b:
            f1_score(
                a,
                b,
                zero_division=0
            ),

    "MCC":
        matthews_corrcoef
}


ci_rows = []

for metric_name, fn in (
    bootstrap_metrics.items()
):

    mean, low, high = (
        bootstrap_metric_ci(
            fn,
            y_test,
            qml_pred
        )
    )

    ci_rows.append({

        "Metric":
            metric_name,

        "Mean":
            mean,

        "CI_Lower_95":
            low,

        "CI_Upper_95":
            high
    })


ci_df = pd.DataFrame(
    ci_rows
)

print("\n" + "=" * 70)
print("QML 95% BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 70)

display(
    ci_df.round(4)
)


# ================================================================
# 28. GRAPH 1
# DATASET CLASS DISTRIBUTION
# ================================================================

plt.figure(
    figsize=(7, 4)
)

counts = (
    scenario_df["Label"]
    .value_counts()
    .sort_index()
)

plt.bar(
    [
        "VIABLE",
        "IMPACTED"
    ],
    counts.values
)

plt.title(
    "Digital-Twin Scenario Class Distribution"
)

plt.ylabel(
    "Number of Scenarios"
)

plt.tight_layout()
plt.show()


# ================================================================
# 29. GRAPH 2
# QML TRAINING CONVERGENCE
# ================================================================

plt.figure(
    figsize=(8, 4)
)

plt.plot(
    range(
        1,
        len(loss_history) + 1
    ),
    loss_history,
    marker="o",
    markersize=3
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "QML Variational Training Convergence"
)

plt.grid(
    True,
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ================================================================
# 30. GRAPH 3
# PCA EXPLAINED VARIANCE
# ================================================================

plt.figure(
    figsize=(8, 4)
)

cumulative_variance = np.cumsum(
    pca.explained_variance_ratio_
)

plt.plot(
    range(
        1,
        N_QUBITS + 1
    ),
    cumulative_variance,
    marker="o"
)

plt.xlabel(
    "Principal Components"
)

plt.ylabel(
    "Cumulative Explained Variance"
)

plt.title(
    "PCA Variance Retained for Quantum Encoding"
)

plt.ylim(
    0,
    1.05
)

plt.grid(
    True,
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ================================================================
# 31. GRAPH 4
# QML CONFUSION MATRIX
# ================================================================

fig, ax = plt.subplots(
    figsize=(6, 5)
)

image = ax.imshow(
    qml_cm,
    cmap="Blues"
)

plt.colorbar(
    image,
    ax=ax
)

labels = [
    "VIABLE",
    "IMPACTED"
]

ax.set_xticks(
    [0, 1],
    labels
)

ax.set_yticks(
    [0, 1],
    labels
)

ax.set_xlabel(
    "Predicted Class"
)

ax.set_ylabel(
    "Actual Class"
)

ax.set_title(
    "QML Confusion Matrix"
)

for i in range(2):

    for j in range(2):

        ax.text(
            j,
            i,
            str(
                qml_cm[i, j]
            ),
            ha="center",
            va="center",
            fontsize=15
        )

plt.tight_layout()
plt.show()


# ================================================================
# 32. GRAPH 5
# NORMALIZED CONFUSION MATRIX
# ================================================================

fig, ax = plt.subplots(
    figsize=(6, 5)
)

image = ax.imshow(
    qml_cm_normalized,
    cmap="Blues",
    vmin=0,
    vmax=1
)

plt.colorbar(
    image,
    ax=ax
)

ax.set_xticks(
    [0, 1],
    labels
)

ax.set_yticks(
    [0, 1],
    labels
)

ax.set_xlabel(
    "Predicted"
)

ax.set_ylabel(
    "Actual"
)

ax.set_title(
    "Normalized QML Confusion Matrix"
)

for i in range(2):

    for j in range(2):

        ax.text(
            j,
            i,
            f"{qml_cm_normalized[i,j]*100:.1f}%",
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()


# ================================================================
# 33. GRAPH 6
# ROC CURVE
# ================================================================

plt.figure(
    figsize=(8, 5)
)

for name, prob in probabilities.items():

    fpr, tpr, _ = (
        roc_curve(
            y_test,
            prob
        )
    )

    auc_value = safe_roc_auc(
        y_test,
        prob
    )

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=(
            f"{name} "
            f"(AUC={auc_value:.3f})"
        )
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curve Comparison"
)

plt.legend()

plt.grid(
    True,
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ================================================================
# 34. GRAPH 7
# PRECISION-RECALL CURVE
# ================================================================

plt.figure(
    figsize=(8, 5)
)

for name, prob in probabilities.items():

    precision_curve, recall_curve, _ = (
        precision_recall_curve(
            y_test,
            prob
        )
    )

    ap_value = safe_pr_auc(
        y_test,
        prob
    )

    plt.plot(
        recall_curve,
        precision_curve,
        linewidth=2,
        label=(
            f"{name} "
            f"(AP={ap_value:.3f})"
        )
    )

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.title(
    "Precision-Recall Curve Comparison"
)

plt.legend()

plt.grid(
    True,
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ================================================================
# 35. GRAPH 8
# MODEL PERFORMANCE COMPARISON
# ================================================================

performance_metrics = [

    "Accuracy",

    "Balanced_Accuracy",

    "Precision",

    "Recall_Sensitivity",

    "Specificity",

    "F1",

    "ROC_AUC"
]

plot_df = (
    metrics_df
    .set_index("Model")[
        performance_metrics
    ]
)

ax = plot_df.plot(
    kind="bar",
    figsize=(12, 5)
)

ax.set_ylim(
    0,
    1.05
)

ax.set_ylabel(
    "Score"
)

ax.set_title(
    "QML vs Classical Baseline Performance"
)

plt.xticks(
    rotation=0
)

plt.legend(
    loc="lower right"
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ================================================================
# 36. GRAPH 9
# RUNTIME COMPARISON
# ================================================================

runtime_df = metrics_df[
    [
        "Model",
        "Train_Time_s",
        "Inference_ms_per_sample"
    ]
].copy()

x = np.arange(
    len(runtime_df)
)

fig, ax1 = plt.subplots(
    figsize=(11, 5)
)

ax1.bar(
    x - 0.18,
    runtime_df[
        "Train_Time_s"
    ],
    width=0.36,
    label="Training Time (s)"
)

ax1.set_ylabel(
    "Training Time (s)"
)

ax1.set_xticks(
    x
)

ax1.set_xticklabels(
    runtime_df["Model"]
)

ax2 = ax1.twinx()

ax2.bar(
    x + 0.18,
    runtime_df[
        "Inference_ms_per_sample"
    ],
    width=0.36,
    alpha=0.65,
    label="Inference ms/sample"
)

ax2.set_ylabel(
    "Inference Time (ms/sample)"
)

ax1.set_title(
    "Training and Inference Runtime"
)

plt.tight_layout()
plt.show()


# ================================================================
# 37. GRAPH 10
# RANDOM FOREST FEATURE IMPORTANCE
# ================================================================

rf = models[
    "Random Forest"
]

# Reconstruct transformed feature names
try:

    feature_names = (
        list(
            numeric_features
        )

        +

        list(
            preprocessor
            .named_transformers_[
                "categorical"
            ]
            .get_feature_names_out(
                categorical_features
            )
        )
    )

    rf_importance = pd.Series(
        rf.feature_importances_,
        index=feature_names
    ).sort_values()

    plt.figure(
        figsize=(10, 7)
    )

    rf_importance.tail(
        15
    ).plot(
        kind="barh"
    )

    plt.xlabel(
        "Importance"
    )

    plt.title(
        "Random Forest Feature Importance"
    )

    plt.tight_layout()
    plt.show()

except Exception as e:

    print(
        "Feature importance plot skipped:",
        e
    )


# ================================================================
# 38. QUANTUM CIRCUIT VISUALIZATION
# ================================================================

print("\n" + "=" * 70)
print("QUANTUM CIRCUIT")
print("=" * 70)

print(

    qml.draw(
        qml_circuit
    )(
        qnp.array(
            X_test_q[0],
            requires_grad=False
        ),
        weights
    )
)


# ================================================================
# 39. TEST PREDICTION EXAMPLES
# ================================================================

prediction_examples = pd.DataFrame({

    "Route_ID":
        scenario_df.iloc[
            test_idx
        ]["Route_ID"].values,

    "Actual":
        np.where(
            y_test == 1,
            "IMPACTED",
            "VIABLE"
        ),

    "QML_Probability_Impacted":
        qml_prob,

    "QML_Prediction":
        np.where(
            qml_pred == 1,
            "IMPACTED",
            "VIABLE"
        )
})


print("\n" + "=" * 70)
print("QML TEST PREDICTION EXAMPLES")
print("=" * 70)

display(
    prediction_examples.head(
        15
    ).round(4)
)


# ================================================================
# 40. SAVE RESULTS
# ================================================================

OUTPUT_DIR = (
    "/content/"
    "PNT1_QML_RESULTS"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# Save scenario dataset
scenario_df.to_csv(
    f"{OUTPUT_DIR}/scenario_dataset.csv",
    index=False
)


# Save model metrics
metrics_df.to_csv(
    f"{OUTPUT_DIR}/model_metrics.csv",
    index=False
)


# Save confidence intervals
ci_df.to_csv(
    f"{OUTPUT_DIR}/qml_confidence_intervals.csv",
    index=False
)


# Save test predictions
prediction_examples.to_csv(
    f"{OUTPUT_DIR}/qml_test_predictions.csv",
    index=False
)


# Save confusion matrix
pd.DataFrame(
    qml_cm,
    index=[
        "Actual_VIABLE",
        "Actual_IMPACTED"
    ],
    columns=[
        "Predicted_VIABLE",
        "Predicted_IMPACTED"
    ]
).to_csv(
    f"{OUTPUT_DIR}/qml_confusion_matrix.csv"
)


# Save QML parameters
np.savez(
    f"{OUTPUT_DIR}/qml_parameters.npz",

    weights=np.asarray(
        weights
    ),

    scale=np.asarray(
        float(
            scale_param
        )
    ),

    bias=np.asarray(
        float(
            bias_param
        )
    ),

    angle_scale=angle_scale
)


# Save preprocessing objects
joblib.dump(
    preprocessor,
    f"{OUTPUT_DIR}/preprocessor.joblib"
)

joblib.dump(
    pca,
    f"{OUTPUT_DIR}/pca.joblib"
)


# Save classical models
for name, model in models.items():

    safe_name = (

        name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "(",
            ""
        )
        .replace(
            ")",
            ""
        )
    )

    joblib.dump(
        model,
        f"{OUTPUT_DIR}/"
        f"{safe_name}.joblib"
    )


# Save training history
pd.DataFrame({

    "Epoch":
        range(
            1,
            len(loss_history) + 1
        ),

    "Loss":
        loss_history

}).to_csv(
    f"{OUTPUT_DIR}/qml_training_history.csv",
    index=False
)


# ================================================================
# 41. SAVE SUMMARY JSON
# ================================================================

qml_metrics = (
    metrics_df[
        metrics_df["Model"]
        == "QML-VQC"
    ]
    .iloc[0]
    .to_dict()
)

summary = {

    "project":
        "PNT1 Logistics Digital Twin",

    "task":
        "Route Impact Classification",

    "qml_model":
        "4-qubit Variational Quantum Classifier",

    "qubits":
        N_QUBITS,

    "variational_layers":
        N_LAYERS,

    "epochs":
        EPOCHS,

    "route_count":
        int(
            transport[
                "Route_ID"
            ].nunique()
        ),

    "disruption_count":
        int(
            len(disruptions)
        ),

    "order_count":
        int(
            len(orders)
        ),

    "scenario_samples":
        int(
            len(scenario_df)
        ),

    "test_routes":
        sorted(
            groups_test.unique()
        ),

    "qml_metrics":
        qml_metrics,

    "note":
        (
            "The supplied logistics workbook is synthetic. "
            "Scenario labels are generated from explicit "
            "operational disruption rules and should not "
            "be interpreted as real-world ground truth."
        )
}

with open(
    f"{OUTPUT_DIR}/summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        default=str
    )


# ================================================================
# 42. ZIP ALL RESULTS
# ================================================================

ZIP_PATH = (
    "/content/"
    "PNT1_QML_RESULTS.zip"
)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for root, dirs, file_list in os.walk(
        OUTPUT_DIR
    ):

        for file_name in file_list:

            full_path = os.path.join(
                root,
                file_name
            )

            arc_name = os.path.relpath(
                full_path,
                OUTPUT_DIR
            )

            z.write(
                full_path,
                arc_name
            )


# ================================================================
# 43. FINAL REPORT
# ================================================================

print("\n")
print("=" * 80)
print("FINAL PNT1 QML REPORT")
print("=" * 80)

for key in [

    "Accuracy",

    "Balanced_Accuracy",

    "Precision",

    "Recall_Sensitivity",

    "Specificity",

    "NPV",

    "F1",

    "ROC_AUC",

    "PR_AUC",

    "MCC",

    "Cohen_Kappa",

    "FPR",

    "FNR",

    "Log_Loss",

    "Brier_Score"

]:

    value = qml_metrics[key]

    print(
        f"{key:25s}: "
        f"{value:.4f}"
    )


print(
    f"\nQML training time     : "
    f"{qml_train_time:.4f} seconds"
)

print(
    f"QML inference/sample  : "
    f"{qml_inference_ms:.4f} ms"
)

print(
    "\nResults folder:"
)

print(
    OUTPUT_DIR
)

print(
    "\nZIP file:"
)

print(
    ZIP_PATH
)

print(
    "\nDONE."
)


# ================================================================
# 44. OPTIONAL COLAB DOWNLOAD
# ================================================================

try:

    from google.colab import files

    print(
        "\nDownloading results ZIP..."
    )

    files.download(
        ZIP_PATH
    )

except Exception:

    pass